# Inspect HSTU-style aggregation and train a recommendation head

Browser Python lab: run in order; cells share variables. Charts come from the code you execute.


## 1 · Temporal data and targets

Use synthetic cycles of three items, with IDs starting at zero and no padding. This easy task does not establish generalization to real users.

In [ ]:
# The blog provides live charts; standalone Python prints chart data.
if 'display_plot' not in globals():
    def display_plot(x, y, title='', xlabel='', ylabel=''):
        print(title, list(zip(x,y)))

import math, random
random.seed(23)
K, D = 3, 3
E = [[float(i==j) for j in range(D)] for i in range(K)]
examples = [([(start+j)%K for j in range(length)],(start+length)%K) for start in range(K) for length in range(1,7)]
print("example history -> target:",examples[:4])
print("embedding shape:",(K,D))

## 2 · Signed SiLU aggregation

Use fixed one-hot embeddings, a distance bias and a residual. Full HSTU projections, normalization and gating are omitted so aggregation is easy to inspect.

In [ ]:
def silu(x): return x/(1+math.exp(-x))
def features(history):
    q=E[history[-1]]
    scores=[sum(a*b for a,b in zip(q,E[item]))-0.3*(len(history)-1-j) for j,item in enumerate(history)]
    weights=[silu(s)/len(history) for s in scores]
    out=[q[d]+sum(a*E[item][d] for a,item in zip(weights,history)) for d in range(D)]
    return out,weights
h,weights=features([0,1,2,0])
print("weights:",weights,"sum:",sum(weights))
print("user vector:",h)
print("Negative weights are allowed: these are not probabilities.")
display_plot(list(range(len(weights))),weights,"Signed aggregation","history position","weight")

## 3 · Train a scoring head

Keep the aggregator fixed and train only D×K scoring parameters. The gradient is h*(p-y). Full end-to-end HSTU-inspired training is in the PyTorch notebook.

In [ ]:
W=[[random.uniform(-0.1,0.1) for _ in range(K)] for _ in range(D)]
def softmax(z):
    e=[math.exp(x-max(z)) for x in z]
    return [x/sum(e) for x in e]
def predict(history):
    h,_=features(history)
    return softmax([sum(h[d]*W[d][j] for d in range(D)) for j in range(K)])
losses=[]
for epoch in range(180):
    grad=[[0.0]*K for _ in range(D)]
    loss=0.0
    for history,target in examples:
        h,_=features(history)
        p=predict(history)
        loss-=math.log(max(p[target],1e-12))/len(examples)
        for d in range(D):
            for j in range(K):
                grad[d][j]+=h[d]*(p[j]-(j==target))/len(examples)
    for d in range(D):
        for j in range(K): W[d][j]-=0.8*grad[d][j]
    losses.append(loss)
print("trained parameters:",D*K)
print("training loss:",losses[0],"->",losses[-1])
display_plot(list(range(180)),losses,"Head training loss","epoch","NLL")

## 4 · Edit the history and inspect ranking

Edit history and inspect the ranking. Longer-sequence tests still follow the same synthetic cycle; they are not independent production data.

In [ ]:
history=[0,1,2,0]
assert history and all(0<=item<K for item in history)
p=predict(history)
print("history:",history)
print("item probabilities:",p)
print("ranking:",sorted(range(K),key=lambda j:-p[j]))
tests=[([(start+j)%K for j in range(8)],(start+8)%K) for start in range(K)]
hits=sum(max(range(K),key=lambda j:predict(h)[j])==target for h,target in tests)
print("Synthetic longer-sequence HR@1:",hits/len(tests))